# Chatbot using Hugging Face Transformers

# Chatbot using Transformers (GPT-2)

This project implements a conversational chatbot using a pre-trained GPT-2 model from Hugging Face.

## Objective

To build a chatbot using a transformer-based model that can generate meaningful responses and interact with users in natural language.

## Install required libraries

In [17]:
!pip install transformers torch --quiet

## Import libraries

In [18]:
# Import required libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Libraries imported successfully.")

Libraries imported successfully.


## Load pre-trained model & tokenizer

In [21]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer from: {MODEL_NAME} ...")
# Load tokenizer — converts raw text to token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from: {MODEL_NAME} ...")
# Load the causal language model (auto-regressive: generates one token at a time)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout, etc. — not training)
model.eval()

print("Model and tokenizer loaded successfully!")

Loading tokenizer from: microsoft/DialoGPT-medium ...
Loading model from: microsoft/DialoGPT-medium ...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In [ ]:
def generate_response(user_input, conversation_history, max_history_turns=5):
    """
    Generate a chatbot response using DialoGPT.

    Parameters:
    -----------
    user_input         : str   — The latest message typed by the user.
    conversation_history: tensor or None — Tensor of previous token IDs (conversation context).
    max_history_turns  : int   — Max number of turns to keep in context to avoid exceeding model limits.

    Returns:
    --------
    response_text      : str   — The chatbot's reply.
    new_history        : tensor — Updated conversation history including this turn.
    """

    # Step 4a: Encode user input and append EOS token (signals end of user's turn)
    # The EOS token acts as a separator between user and bot turns in DialoGPT
    user_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"   # Return as PyTorch tensor
    )

    # Step 4b: Concatenate with existing conversation history
    if conversation_history is not None:
        # Append the new user input to the conversation history along the sequence dimension
        bot_input_ids = torch.cat([conversation_history, user_input_ids], dim=-1)
    else:
        # First turn — no history yet
        bot_input_ids = user_input_ids

    # Step 4c: Trim history if it exceeds model's practical limit
    # DialoGPT supports up to 1024 tokens; we trim from the left to stay within bounds
    max_tokens = 900  # Conservative limit to leave room for new response
    if bot_input_ids.shape[-1] > max_tokens:
        bot_input_ids = bot_input_ids[:, -max_tokens:]

    # Step 4d: Build explicit attention mask
    # DialoGPT uses EOS as pad token, so the model cannot auto-infer the mask.
    # We pass a mask of all 1s (every token is real, none are padding) to
    # suppress the warning and ensure reliable generation results.
    attention_mask = torch.ones_like(bot_input_ids)

    # Step 4e: Generate a response using the model
    with torch.no_grad():  # Disable gradient computation (inference only)
        output_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,   # Explicit mask — fixes pad/eos warning
            max_new_tokens=150,          # Max tokens to generate for the reply
            pad_token_id=tokenizer.eos_token_id,  # Use EOS as padding token
            do_sample=True,              # Enable sampling for varied responses
            top_k=50,                    # Top-K sampling: consider top 50 probable tokens
            top_p=0.92,                  # Nucleus sampling: cumulative probability threshold
            temperature=0.75,            # Controls randomness: lower = more focused replies
            repetition_penalty=1.3,      # Penalize repeating the same tokens
            no_repeat_ngram_size=3,      # Prevent repeating 3-gram phrases
        )

    # Step 4g: Decode only the newly generated tokens (exclude the input prompt)
    # output_ids includes the full input + generated reply; we slice off the input part
    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    # Step 4g: Update conversation history with the full exchange (input + reply)
    new_history = output_ids

    return response_text, new_history


print("Response generation function defined successfully.")

In [ ]:
def run_chatbot():
    """
    Main chatbot loop.
    Maintains multi-turn conversation history and interacts with the user
    via console input until the user types 'exit' or 'quit'.
    """

    print("=" * 60)
    print("   DialoGPT Conversational Chatbot")
    print("   Powered by microsoft/DialoGPT-medium")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to end the conversation.)")
    print("-" * 60)

    # Initialize conversation history as None (no prior context on first turn)
    conversation_history = None
    turn_count = 0  # Track number of conversation turns

    # --- Main conversation loop ---
    while True:
        # Accept user input
        user_input = input("You: ").strip()

        # Handle empty input gracefully
        if not user_input:
            print("Chatbot: I didn't catch that. Could you say something?")
            continue

        # Exit condition: stop when user types 'exit' or 'quit' (case-insensitive)
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 60)
            print("Chatbot: Thank you for chatting! Have a great day. Goodbye!")
            print("=" * 60)
            break

        # Generate a response using the transformer model
        response, conversation_history = generate_response(user_input, conversation_history)

        # Fallback: if the model returns an empty string, provide a default reply
        if not response.strip():
            response = "I'm not sure how to respond to that. Could you rephrase?"

        # Display the chatbot's response
        print(f"Chatbot: {response}")
        print("-" * 60)

        turn_count += 1


# --- Entry Point ---
run_chatbot()